<a href="https://colab.research.google.com/github/19mddill/Machine_Learning_Projects/blob/main/ensemble_learning_and_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [2]:
X,y = make_moons(n_samples=500,noise=0.3,random_state=42)
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [3]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr',LogisticRegression()),
        ('rf',RandomForestClassifier()),
        ('svc',SVC())
    ]
)

In [4]:
voting_clf.fit(X_train,y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [5]:
for name,clf in voting_clf.named_estimators_.items():
    print(name,"=",clf.score(X_test,y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [6]:
voting_clf.predict(X_test[:1])

array([1])

In [7]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

In [8]:
voting_clf.score(X_test,y_test)

0.896

In [9]:
voting_clf.voting = 'soft'
voting_clf.named_estimators["svc"].probability = True
voting_clf.fit(X_train,y_train)


VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()),
                             ('svc', SVC(probability=True))],
                 voting='soft')

In [10]:
for name,clf in voting_clf.named_estimators_.items():
    print(name,"=",clf.score(X_test,y_test))

lr = 0.864
rf = 0.888
svc = 0.896


In [11]:
print("\n=== Probabilities for X_test[0] ===")
print(f"True label: {y_test[0]}")
for name, clf in zip(voting_clf.named_estimators_, voting_clf.estimators_):
    proba = clf.predict_proba(X_test[:1])[0]
    print(f"{name}: class0={proba[0]:.3f}, class1={proba[1]:.3f}")


=== Probabilities for X_test[0] ===
True label: 1
lr: class0=0.499, class1=0.501
rf: class0=0.490, class1=0.510
svc: class0=0.579, class1=0.421


In [12]:
X_test[:1]

array([[0.50169252, 0.21717211]])

In [13]:
[clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_]

[array([0.49900001, 0.50099999]),
 array([0.49, 0.51]),
 array([0.57880978, 0.42119022])]

In [15]:
import numpy as np
np.array([clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_])

array([[0.49900001, 0.50099999],
       [0.49      , 0.51      ],
       [0.57880978, 0.42119022]])

In [16]:

print("\n=== Averaged Probabilities (soft voting logic) ===")
all_probas = np.array([clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_])
avg_proba = all_probas.mean(axis=0)
print(f"class0 avg={avg_proba[0]:.3f}, class1 avg={avg_proba[1]:.3f}")
print(f"Soft vote picks: class {np.argmax(avg_proba)}")


=== Averaged Probabilities (soft voting logic) ===
class0 avg=0.523, class1 avg=0.477
Soft vote picks: class 0


In [17]:
avg_proba

array([0.52260326, 0.47739674])

In [18]:

print(f"\nEnsemble prediction: {voting_clf.predict(X_test[:1])[0]}")
print(f"Ensemble score: {voting_clf.score(X_test, y_test):.4f}")


Ensemble prediction: 0
Ensemble score: 0.9120


# Bagging

In [20]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [19]:
X,y = make_moons(n_samples=500,noise=0.3,random_state=42)
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [24]:
X_train.size

750

In [21]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,
    n_jobs=-1
)

In [25]:
bag_clf.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1)

In [26]:
bag_clf.score(X_test,y_test)

0.92